In [1]:
from fastai.vision.all import *
import pandas as pd, numpy as np, json
from functools import partial

# Fix: missing import caused NameError for StratifiedShuffleSplit
from sklearn.model_selection import StratifiedShuffleSplit



/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
path = Path("/kaggle/input/cassava-leaf-disease-classification/")
path.ls()



(#17) [Path('/kaggle/input/cassava-leaf-disease-classification/train.csv.zip'),Path('/kaggle/input/cassava-leaf-disease-classification/sample_submission.csv.zip'),Path('/kaggle/input/cassava-leaf-disease-classification/test.zip'),Path('/kaggle/input/cassava-leaf-disease-classification/test_tfrecords.zip'),Path('/kaggle/input/cassava-leaf-disease-classification/train_tfrecords.zip'),Path('/kaggle/input/cassava-leaf-disease-classification/test_images.zip'),Path('/kaggle/input/cassava-leaf-disease-classification/train.csv'),Path('/kaggle/input/cassava-leaf-disease-classification/test_images'),Path('/kaggle/input/cassava-leaf-disease-classification/test_tfrecords'),Path('/kaggle/input/cassava-leaf-disease-classification/cassava-leaf-disease-classification'),Path('/kaggle/input/cassava-leaf-disease-classification/train_tfrecords'),Path('/kaggle/input/cassava-leaf-disease-classification/train_images'),Path('/kaggle/input/cassava-leaf-disease-classification/train.zip'),Path('/kaggle/input/cas

In [3]:
train_df = pd.read_csv(path / "train.csv")
with open(path / "label_num_to_disease_map.json") as f:
    label_dict = json.load(f)
label_dict = {int(k): v for k, v in label_dict.items()}

df = train_df.set_index("image_id")
labels = df.to_dict()["label"]


def get_label(labels, x):
    x = Path(x)
    return labels[x.name]


def get_data(labels, bs=64, presize=500, resize=384, valid_pct=0.2, seed=42):
    tfms = [
        Rotate(90),
        Warp(magnitude=0.4, p=1.0),
        Zoom(min_zoom=0.9, max_zoom=1.3),
        Brightness(max_lighting=0.5),
        Flip(),
        Contrast(),
        Resize(resize),
    ]
    comp = setup_aug_tfms(tfms)

    # Fix: remove invalid `splitter=` kwarg passed to `.dataloaders(...)` (would raise if this helper is used)
    return DataBlock(
        blocks=(ImageBlock, CategoryBlock),
        get_items=lambda p: get_image_files(p),
        get_y=partial(get_label, labels),
        item_tfms=[Resize(presize)],
        batch_tfms=[*comp, Normalize.from_stats(*imagenet_stats)],
    ).dataloaders(
        path / "train_images",
        bs=bs,
        num_workers=8,
    )




In [4]:
set_seed(42, reproducible=True)

train_df2 = train_df.copy()
splits = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx = np.arange(len(train_df2))
y = train_df2["label"].values
train_idx, valid_idx = next(splits.split(idx, y))
is_valid = np.zeros(len(train_df2), dtype=bool)
is_valid[valid_idx] = True
train_df2["is_valid"] = is_valid

labels2 = train_df2.set_index("image_id")["label"].to_dict()


def get_data_from_df(train_df, labels, bs=64, presize=384, resize=384):
    tfms = [
        Rotate(90),
        Warp(magnitude=0.4, p=1.0),
        Zoom(min_zoom=0.9, max_zoom=1.3),
        Brightness(max_lighting=0.5),
        Flip(),
        Contrast(),
        Resize(resize),
    ]
    comp = setup_aug_tfms(tfms)

    return DataBlock(
        blocks=(ImageBlock, CategoryBlock),
        get_x=ColReader("image_id", pref=str(path / "train_images") + "/"),
        get_y=ColReader("label"),
        splitter=ColSplitter(col="is_valid"),
        item_tfms=[Resize(presize)],
        batch_tfms=[*comp, Normalize.from_stats(*imagenet_stats)],
    ).dataloaders(train_df, bs=bs, num_workers=8)


dls = get_data_from_df(train_df2, labels2, bs=128, presize=384, resize=384)

learn = cnn_learner(dls, models.resnet50, metrics=[accuracy], pretrained=True)

test_files = get_image_files(path / "test_images")
predictions = []

weights_base = Path("/kaggle/input/casava-starter/models")
for fold in range(1):
    wfile = weights_base / f"resnet50-full-fold_{fold}.pth"
    if wfile.exists():
        state = torch.load(wfile, map_location="cpu")
        if isinstance(state, dict) and "model" in state:
            learn.model.load_state_dict(state["model"])
        elif isinstance(state, dict):
            learn.model.load_state_dict(state)
        else:
            learn.model.load_state_dict(state)
    else:
        learn.fine_tune(4, base_lr=3e-3)

    test_dl = dls.test_dl(test_files)
    preds, _ = learn.tta(dl=test_dl, n=6)
    predictions.append(preds)



/usr/local/lib/python3.11/dist-packages/fastai/vision/learner.py:303: UserWarning: `cnn_learner` has been renamed to `vision_learner` -- please update your code
  warn("`cnn_learner` has been renamed to `vision_learner` -- please update your code")
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 113MB/s]


epoch,train_loss,valid_loss,accuracy,time
0,1.189922,0.765673,0.753271,01:29


epoch,train_loss,valid_loss,accuracy,time
0,0.679823,0.534784,0.827236,01:48
1,0.538788,0.393448,0.866489,01:48
2,0.447160,0.363653,0.878238,01:48
3,0.403571,0.360937,0.882777,01:48


In [5]:
if len(predictions) == 0:
    raise RuntimeError(
        "No predictions were generated; check model loading/training step."
    )

mean_preds = torch.mean(torch.stack(predictions), dim=0)
preds = torch.argmax(mean_preds, dim=1)

test_fn = [p.name for p in test_files]

assert len(test_fn) == len(
    preds
), f"Mismatch: {len(test_fn)} filenames vs {len(preds)} predictions"



In [6]:
submission = pd.DataFrame(
    {"image_id": test_fn, "label": preds.cpu().numpy().astype(int)}
)
submission.to_csv("submission.csv", index=False)
submission.head()



,image_id,label
0,2574872277.jpg,3
1,1449210447.jpg,3
2,1269550303.jpg,4
3,1411799307.jpg,3
4,2474336811.jpg,3


In [7]:
submission

,image_id,label
0,2574872277.jpg,3
1,1449210447.jpg,3
2,1269550303.jpg,4
3,1411799307.jpg,3
4,2474336811.jpg,3
...,...,...
2671,2551245450.jpg,4
2672,2576527162.jpg,3
2673,1244824731.jpg,3
2674,1368056046.jpg,3
